# 서울시 한식음식점 상권 성장 분석 

## 01 | CSV 파일 불러오기


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

# 추정매출 CSV
sales23 = pd.read_csv('서울시_상권분석서비스(추정매출-상권)_2023년.csv', encoding='cp949', low_memory=False)
sales24 = pd.read_csv('서울시 상권분석서비스(추정매출-상권)_2024년.csv', encoding='cp949', low_memory=False)
sales25 = pd.read_csv('서울시 상권분석서비스(추정매출-상권)_2025년.csv', encoding='cp949', low_memory=False)

# 점포 CSV
store23 = pd.read_csv('서울시_상권분석서비스(점포-상권)_2023년.csv', encoding='cp949', low_memory=False)
store24 = pd.read_csv('서울시 상권분석서비스(점포-상권)_2024년.csv', encoding='cp949', low_memory=False)
store25 = pd.read_csv('서울시 상권분석서비스(점포-상권)_2025년.csv', encoding='cp949', low_memory=False)


## 02 | 점포 컬럼명 통일 후 한식음식점만 추출

In [ ]:
# 영문/한글 점포 컬럼명 통일
store_names = {
    'stdr_yyqu_cd': '기준_년분기_코드',
    'trdar_cd': '상권_코드',
    'trdar_cd_nm': '상권_코드_명',
    'svc_induty_cd': '서비스_업종_코드',
    'stor_co': '점포_수',
    'STDR_YYQU_CD': '기준_년분기_코드',
    'TRDAR_CD': '상권_코드',
    'TRDAR_CD_NM': '상권_코드_명',
    'SVC_INDUTY_CD': '서비스_업종_코드',
    'STOR_CO': '점포_수'
}
store23 = store23.rename(columns=store_names)
store24 = store24.rename(columns=store_names)
store25 = store25.rename(columns=store_names)

# 필요한 컬럼만 사용
sales_cols = ['기준_년분기_코드', '상권_코드', '상권_코드_명', '서비스_업종_코드',
              '당월_매출_금액', '시간대_11~14_매출_금액',
              '시간대_17~21_매출_금액', '시간대_21~24_매출_금액']
store_cols = ['기준_년분기_코드', '상권_코드', '서비스_업종_코드', '점포_수']

sales = pd.concat([sales23, sales24, sales25], ignore_index=True)
stores = pd.concat([store23, store24, store25], ignore_index=True)

# 컬럼명이 다르면 누락된 컬럼을 정확히 안내
missing_sales = pd.Index(sales_cols).difference(sales.columns)
missing_stores = pd.Index(store_cols).difference(stores.columns)
if not missing_sales.empty or not missing_stores.empty:
    raise ValueError(f'컬럼명 확인 필요 | 추정매출: {list(missing_sales)} / 점포: {list(missing_stores)}')

sales = sales.loc[sales['서비스_업종_코드'].astype('string').str.strip().eq('CS100001'), sales_cols].copy()
stores = stores.loc[stores['서비스_업종_코드'].astype('string').str.strip().eq('CS100001'), store_cols].copy()

# 상권코드는 문자열, 분기코드는 숫자로 통일
sales['상권_코드'] = sales['상권_코드'].astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
stores['상권_코드'] = stores['상권_코드'].astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
sales['기준_년분기_코드'] = pd.to_numeric(sales['기준_년분기_코드'], errors='coerce')
stores['기준_년분기_코드'] = pd.to_numeric(stores['기준_년분기_코드'], errors='coerce')

# 코드가 없는 행만 제외. 정수형 변환은 결측 제거 후 진행
sales = sales.dropna(subset=['상권_코드', '기준_년분기_코드']).copy()
stores = stores.dropna(subset=['상권_코드', '기준_년분기_코드']).copy()
sales['기준_년분기_코드'] = sales['기준_년분기_코드'].astype('int64')
stores['기준_년분기_코드'] = stores['기준_년분기_코드'].astype('int64')


## 03 | 데이터 결합 및 파생변수

- **점심:** 11~14시(3시간)
- **저녁·야간:** 17~21시 + 21~24시(7시간)
- 점심과 저녁·야간은 길이가 다르므로 **두 시간대의 총매출끼리 직접 비교하지 않는다.**

In [ ]:
# 중복 여부 확인: 원본 중복을 임의로 삭제하지 않음
key = ['상권_코드', '기준_년분기_코드']
if sales.duplicated(key).any() or stores.duplicated(key).any():
    raise ValueError('상권코드·분기코드 중복 행이 있습니다. 원본 데이터를 확인하세요.')

# 숫자형 변환
amount_cols = ['당월_매출_금액', '시간대_11~14_매출_금액',
               '시간대_17~21_매출_금액', '시간대_21~24_매출_금액']
sales[amount_cols] = sales[amount_cols].apply(pd.to_numeric, errors='coerce')
stores['점포_수'] = pd.to_numeric(stores['점포_수'], errors='coerce')

# 매출을 기준으로 점포 수 연결
panel = sales.merge(stores[key + ['점포_수']], on=key, how='left', validate='one_to_one')
panel['연도'] = panel['기준_년분기_코드'] // 10
panel['분기'] = panel['기준_년분기_코드'] % 10
panel = panel.loc[panel['연도'].isin([2023, 2024, 2025]) & panel['분기'].between(1, 4)].copy()

# 필수 자료인 전체 매출·점포수가 없는 행은 제외 (시간대 자료 누락만으로 전체 행을 버리지는 않음)
print('2025년 점포수 누락 행:',
      panel.loc[panel['연도'].eq(2025), '점포_수'].isna().sum())
panel = panel.dropna(subset=['당월_매출_금액', '점포_수']).copy()

# 분기 환산 추정매출 (억원)
panel['전체매출'] = panel['당월_매출_금액'] * 3 / 1e8
panel['점심매출'] = panel['시간대_11~14_매출_금액'] * 3 / 1e8
panel['저녁매출'] = (panel['시간대_17~21_매출_금액'] + panel['시간대_21~24_매출_금액']) * 3 / 1e8

print('연도·분기별 한식음식점 상권 수')
display(pd.crosstab(panel['연도'], panel['분기']))

if not {2024, 2025}.issubset(set(panel['연도'])):
    raise ValueError('2024년 또는 2025년 자료가 없습니다. CSV 결합 상태를 확인하세요.')

## 04 | 2024~2025년 8개 분기 및 연간 지표

**각 연도 1~4분기**가 모두 존재하는 상권만 남긴다. 분기별 YoY는 2025년 같은 분기와 2024년을 비교하며, 연간 지표는 4개 분기 합계 / 점포수 연평균으로 계산한다.

In [ ]:
# 2024·2025년 분기 자료가 모두 있는 상권
main = panel.loc[panel['연도'].isin([2024, 2025])].copy()
quarter_counts = main.groupby(['상권_코드', '연도'])['분기'].nunique().unstack(fill_value=0)
quarter_counts = quarter_counts.reindex(columns=[2024, 2025], fill_value=0)
valid_codes = quarter_counts.index[quarter_counts[2024].eq(4) & quarter_counts[2025].eq(4)]
main = main.loc[main['상권_코드'].isin(valid_codes)].sort_values(['상권_코드', '연도', '분기']).copy()

print('8개 분기 모두 관측된 상권:', main['상권_코드'].nunique())
if main.empty:
    raise ValueError('8개 분기가 모두 있는 상권이 0개입니다. 2025년 점포 컬럼명과 결합 상태를 확인하세요.')

# 2024→2025년 같은 분기 YoY (8개 분기가 정렬된 상태에서 4행 전과 비교)
main['전체_YoY'] = main.groupby('상권_코드')['전체매출'].pct_change(4, fill_method=None) * 100
main['점심_YoY'] = main.groupby('상권_코드')['점심매출'].pct_change(4, fill_method=None) * 100
main['저녁_YoY'] = main.groupby('상권_코드')['저녁매출'].pct_change(4, fill_method=None) * 100
main[['전체_YoY', '점심_YoY', '저녁_YoY']] = (
    main[['전체_YoY', '점심_YoY', '저녁_YoY']].replace([np.inf, -np.inf], np.nan)
)

# 연간 추정매출 합계: 분기값 하나라도 누락되면 해당 연간 지표는 결측 처리
annual_sales = main.groupby(['상권_코드', '연도'])[['전체매출', '점심매출', '저녁매출']].sum(min_count=4)
annual_stores = main.groupby(['상권_코드', '연도'])['점포_수'].mean().rename('평균점포수')
annual = annual_sales.join(annual_stores).reset_index()

# 연도별 집계를 명확한 컬럼명으로 연결
annual24 = annual.loc[annual['연도'].eq(2024)].drop(columns='연도').set_index('상권_코드').add_suffix('_2024')
annual25 = annual.loc[annual['연도'].eq(2025)].drop(columns='연도').set_index('상권_코드').add_suffix('_2025')
compare = annual24.join(annual25, how='inner', validate='one_to_one')

# 연간 성장률 (%)과 점포당 연간 추정매출 (억원)
compare['전체성장률'] = (compare['전체매출_2025'] / compare['전체매출_2024'].replace(0, np.nan) - 1) * 100
compare['점심성장률'] = (compare['점심매출_2025'] / compare['점심매출_2024'].replace(0, np.nan) - 1) * 100
compare['저녁성장률'] = (compare['저녁매출_2025'] / compare['저녁매출_2024'].replace(0, np.nan) - 1) * 100
compare['점포당_2024'] = compare['전체매출_2024'] / compare['평균점포수_2024'].replace(0, np.nan)
compare['점포당_2025'] = compare['전체매출_2025'] / compare['평균점포수_2025'].replace(0, np.nan)
compare['점포당성장률'] = (compare['점포당_2025'] / compare['점포당_2024'].replace(0, np.nan) - 1) * 100
compare['점심점포당_2025'] = compare['점심매출_2025'] / compare['평균점포수_2025'].replace(0, np.nan)
compare['저녁점포당_2025'] = compare['저녁매출_2025'] / compare['평균점포수_2025'].replace(0, np.nan)
compare['점심비중_2025'] = compare['점심매출_2025'] / compare['전체매출_2025'].replace(0, np.nan) * 100
compare['상권명'] = main.groupby('상권_코드')['상권_코드_명'].last()

display(compare[['상권명', '평균점포수_2024', '평균점포수_2025',
                 '전체성장률', '점포당성장률']].head().round(2))

## 05 | 성장 상권 120개 선별 및 최근 성장률 변화

**선정 조건**

1. 2024·2025년 각각 4개 분기 관측, 평균 점포수 **10개 이상**
2. 2024년 전체 연간 추정매출 **0보다 큼**
3. 2025년 연간 전체 매출과 점포당 연간 매출이 **모두 증가**
4. 2025년 **1~4분기 전체 매출 YoY가 모두 양수**

2025년 3분기 YoY와 4분기 YoY의 차이를 **%p**로 표시한다. 이는 QoQ(직전 분기 대비 매출 변화)가 아니다.

In [ ]:
q25 = main.loc[main['연도'].eq(2025)].copy()

# 네 분기 모두 YoY가 양수인지 확인 (결측 또는 0을 포함하면 미충족)
yoy_cols = ['전체_YoY', '점심_YoY', '저녁_YoY']
yoy_min = q25.groupby('상권_코드')[yoy_cols].min()
yoy_count = q25.groupby('상권_코드')[yoy_cols].count()
positive = yoy_min.gt(0) & yoy_count.eq(4)

eligible = compare.loc[
    compare['평균점포수_2024'].ge(10)
    & compare['평균점포수_2025'].ge(10)
    & compare['전체매출_2024'].gt(0)
].copy()

growth = eligible.loc[
    eligible['전체성장률'].gt(0)
    & eligible['점포당성장률'].gt(0)
    & positive.reindex(eligible.index, fill_value=False)['전체_YoY']
].copy()

q3 = q25.loc[q25['분기'].eq(3)].set_index('상권_코드')
q4 = q25.loc[q25['분기'].eq(4)].set_index('상권_코드')
growth['Q3_YoY'] = q3['전체_YoY']
growth['Q4_YoY'] = q4['전체_YoY']
growth['최근변화_pp'] = growth['Q4_YoY'] - growth['Q3_YoY']
growth['최근흐름'] = np.where(growth['최근변화_pp'].ge(0), '개선·유지', '둔화')

print('기본 분석 조건 충족:', len(eligible))
print('지속 성장 상권:', len(growth))
print('최근 성장률 변화:')
display(growth['최근흐름'].value_counts())

## 06 | 영업 유형별 비교 후보 선별

성장 상권 안에서 **점심 / 저녁·야간** 각각 2025년 네 분기 YoY가 양수인 상권을 구한다.

그다음 **각 유형별 후보군 내부**에서 성장률과 2025년 점포당 추정매출이 **모두 중앙값 이상**인 비교 후보를 선정한다. 한 상권이 여러 유형에 포함될 수 있다.

In [ ]:
lunch_pool = growth.loc[positive.reindex(growth.index, fill_value=False)['점심_YoY']].copy()
dinner_pool = growth.loc[positive.reindex(growth.index, fill_value=False)['저녁_YoY']].copy()

all_candidates = growth.loc[
    growth['전체성장률'].ge(growth['전체성장률'].median())
    & growth['점포당_2025'].ge(growth['점포당_2025'].median())
].copy()
lunch_candidates = lunch_pool.loc[
    lunch_pool['점심성장률'].ge(lunch_pool['점심성장률'].median())
    & lunch_pool['점심점포당_2025'].ge(lunch_pool['점심점포당_2025'].median())
].copy()
dinner_candidates = dinner_pool.loc[
    dinner_pool['저녁성장률'].ge(dinner_pool['저녁성장률'].median())
    & dinner_pool['저녁점포당_2025'].ge(dinner_pool['저녁점포당_2025'].median())
].copy()

summary = pd.DataFrame({
    '유형': ['종합 성장형', '점심 성장형', '저녁·야간 성장형'],
    '성장 조건 충족': [len(growth), len(lunch_pool), len(dinner_pool)],
    '추가 비교 후보': [len(all_candidates), len(lunch_candidates), len(dinner_candidates)]
})
display(summary)

## 07 | 최종 후보 3곳 정량 검증

최종 후보는 **수익성 순위가 아니라 영업 전략별 대표적인 입점 검토 후보**다. 데이터와 기존 포트폴리오가 일치하는지 확인한다.

In [ ]:
codes = ['3120243', '3120025', '3111000']
missing = pd.Index(codes).difference(growth.index)
if len(missing) > 0:
    raise ValueError(f'지속 성장 상권에 없는 최종 후보: {list(missing)}. 앞 셀의 2025년 자료와 선정 조건을 확인하세요.')

# 한 유형당 한 행: 성장률·점포당 매출·Q3/Q4 성장률은 해당 영업 시간대를 사용
final = pd.DataFrame({
    '상권': ['천호역', '롯데백화점(시청광장 지하쇼핑센터)', '대왕초등학교(세곡동사거리서남측)'],
    '유형': ['종합 성장형', '점심 성장형', '저녁·야간 성장형'],
    '2025 연간 성장률(%)': [growth.at['3120243', '전체성장률'],
                          growth.at['3120025', '점심성장률'],
                          growth.at['3111000', '저녁성장률']],
    '2025 점포당 추정매출(억원)': [growth.at['3120243', '점포당_2025'],
                              growth.at['3120025', '점심점포당_2025'],
                              growth.at['3111000', '저녁점포당_2025']],
    'Q3 YoY(%)': [q3.at['3120243', '전체_YoY'],
                  q3.at['3120025', '점심_YoY'],
                  q3.at['3111000', '저녁_YoY']],
    'Q4 YoY(%)': [q4.at['3120243', '전체_YoY'],
                  q4.at['3120025', '점심_YoY'],
                  q4.at['3111000', '저녁_YoY']],
    '2024 평균점포수': [growth.at['3120243', '평균점포수_2024'],
                     growth.at['3120025', '평균점포수_2024'],
                     growth.at['3111000', '평균점포수_2024']],
    '2025 평균점포수': [growth.at['3120243', '평균점포수_2025'],
                     growth.at['3120025', '평균점포수_2025'],
                     growth.at['3111000', '평균점포수_2025']]
})
final['최근변화(%p)'] = final['Q4 YoY(%)'] - final['Q3 YoY(%)']
final['점포수증감률(%)'] = (final['2025 평균점포수'] / final['2024 평균점포수'] - 1) * 100

display(final.round(2))
print('롯데백화점 점심 매출 비중:', round(growth.at['3120025', '점심비중_2025'], 2), '%')
print('유형별 비교 후보 포함: ',
      '천호역', '3120243' in all_candidates.index,
      '/ 롯데', '3120025' in lunch_candidates.index,
      '/ 대왕초', '3111000' in dinner_candidates.index)

## 08 | 대왕초등학교 기저효과 추가 검증

2024년 1분기 매출이 낮아 연간 성장률을 크게 보이게 했을 가능성을 확인한다. **1분기를 빼고 2~4분기 합계끼리 비교**한다.

In [ ]:
daewang = main.loc[main['상권_코드'].eq('3111000')].copy()
daewang24 = daewang.loc[daewang['연도'].eq(2024)]
daewang25 = daewang.loc[daewang['연도'].eq(2025)]

if daewang24['분기'].nunique() != 4 or daewang25['분기'].nunique() != 4:
    raise ValueError('대왕초등학교의 2024·2025년 분기 자료가 완전하지 않습니다.')

base_check = pd.DataFrame({
    '기간': ['1~4분기 전체', '2~4분기 (1분기 제외)'],
    '2024 매출(억원)': [daewang24['저녁매출'].sum(),
                        daewang24.loc[daewang24['분기'].isin([2, 3, 4]), '저녁매출'].sum()],
    '2025 매출(억원)': [daewang25['저녁매출'].sum(),
                        daewang25.loc[daewang25['분기'].isin([2, 3, 4]), '저녁매출'].sum()]
})
base_check['성장률(%)'] = (base_check['2025 매출(억원)'] / base_check['2024 매출(억원)'] - 1) * 100

display(base_check.round(2))

## 09 | 시각화

In [ ]:
# 한강로동땡땡거리 vs 천호역 분기별 성장률 비교

trend = q25[q25['상권_코드'].isin(['3120243', '3110062'])]

trend = trend.pivot(
    index='분기',
    columns='상권_코드',
    values='전체_YoY'
)

plt.figure(figsize=(9, 5))

plt.plot(trend.index, trend['3120243'], marker='o', label='천호역')
plt.plot(trend.index, trend['3110062'], marker='o', label='한강로동땡땡거리')

plt.title('2025년 분기별 YoY 성장률 비교')
plt.xlabel('분기')
plt.ylabel('성장률 (%)')
plt.xticks([1, 2, 3, 4])
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# 천호역 - 전체 매출
cheonho = main[main['상권_코드'] == '3120243']
cheonho = cheonho.pivot(index='분기', columns='연도', values='전체매출')

cheonho.plot(marker='o', figsize=(8, 5))
plt.title('천호역 | 분기별 전체 추정매출')
plt.xlabel('분기')
plt.ylabel('추정매출 (억원)')
plt.xticks([1, 2, 3, 4])
plt.show()

In [ ]:
# 롯데백화점 - 점심 매출
lotte = main[main['상권_코드'] == '3120025']
lotte = lotte.pivot(index='분기', columns='연도', values='점심매출')

lotte.plot(marker='o', figsize=(8, 5))
plt.title('롯데백화점 | 분기별 점심 추정매출')
plt.xlabel('분기')
plt.ylabel('추정매출 (억원)')
plt.xticks([1, 2, 3, 4])
plt.show()

In [ ]:
# 대왕초등학교 - 저녁·야간 매출
daewang = main[main['상권_코드'] == '3111000']
daewang = daewang.pivot(index='분기', columns='연도', values='저녁매출')

daewang.plot(marker='o', figsize=(8, 5))
plt.title('대왕초등학교 | 분기별 저녁·야간 추정매출')
plt.xlabel('분기')
plt.ylabel('추정매출 (억원)')
plt.xticks([1, 2, 3, 4])
plt.show()